In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

save_dir = "data/xenium-tcr-kidney/processed"

import matplotlib.pyplot as plt
import nichepca as npc
import numpy as np
import pandas as pd
import scanpy as sc

from spatial_tcr.clustering import leiden_unique

# Set the verbosity level
# sc.settings.verbosity = 3

In [ ]:
def cat_to_indicator(df, col, prefix="is_"):
    # Example of transforming categorical column to indicator columns
    series = df[col].astype(str)
    indicator_df = pd.get_dummies(series).multiply(series, axis=0)
    indicator_df = indicator_df.replace("", np.nan)
    # rename columns to is_<category>
    indicator_df.columns = [f"{prefix}{col}" for col in indicator_df.columns]
    return indicator_df

In [ ]:
path = "data/xenium/processed/04.1-kidney_tcr_tsub_harmonized.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()
# adata.obs["sample"] = [name.split("_output-")[-1] for name in adata.obs_names]
adata

In [ ]:
adata.obs["cell_type_l2"].value_counts()

In [ ]:
ct_col = "cell_type_l2"
ct_df = cat_to_indicator(adata.obs, "cell_type_l1", prefix="is_")
adata.obs[ct_df.columns] = ct_df

In [ ]:
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

In [ ]:
adata.obs[ct_col].nunique()

In [ ]:
radius = 20
# radius = None

knn = None
# knn = 20

adata.X = adata.layers["counts"].copy()
npc.wf.nichepca(
    adata, knn=knn, radius=radius, obs_key=ct_col, sample_key="sample", n_comps=30
)
adata.X = adata.layers["counts"].copy()

In [ ]:
leiden_unique(adata, resolution=0.5, use_rep="X_npca", n_neighbors=50)
adata.obs.leiden.nunique()

In [ ]:
# insert original clusters for reproducibility
npca_clusters = pd.read_csv("data/xenium/npca_clusters.csv", index_col=0)
adata.obs["leiden"] = adata.obs["leiden"].astype(str)
adata.obs.loc[npca_clusters.index, "leiden"] = npca_clusters["leiden"].astype(str)
adata.obs["leiden"].value_counts()

In [ ]:
markers = {
    "Immune": [
        "CD3E",
        "CD3D",
        "CD4",
        "CD8A",
        "CXCL13",
        "CXCR5",
        "TNF",
        "CD19",
        "MS4A1",
    ],
    "Glom.": [
        "PODXL",
        "PECAM1",
        # "PDGFRB",
    ],
    "tubuloint.": ["UMOD"],
}

In [ ]:
adata.obs["leiden"].value_counts().plot(
    kind="bar", figsize=(5, 5), ylabel="Number of cells"
)
# plt.yscale("log")
# rotate xticks
plt.xticks(rotation=0)

adata.X = adata.layers["log1p"].copy()
sc.pl.dotplot(
    adata,
    var_names=markers,
    groupby="leiden",
    standard_scale="var",
)

In [ ]:
ind_df = cat_to_indicator(adata.obs, "leiden", prefix="leiden_")
adata.obs[ind_df.columns] = ind_df

In [ ]:
glom_clust = 1
glom_key = f"leiden_{glom_clust}"

adata.obs[glom_key] = adata.obs[glom_key].fillna("None")

In [ ]:
for s in adata.obs["sample"].unique():
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(
        ad_sub, color=["is_T", "is_B", "is_POD"], na_color="lightgray", spot_size=10
    )
    sc.pl.spatial(ad_sub, color=glom_key, na_color="lightgray", spot_size=10)
    break

In [ ]:
# plt.rcParams["figure.figsize"] = (15, 15)
# plt.rcParams["figure.dpi"] = 300

for s in adata.obs["sample"].unique():
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(
        ad_sub, color=["is_T", "is_B", "is_POD"], na_color="lightgray", spot_size=10
    )

    not_na_cols = [c for c in ind_df.columns if ad_sub.obs[c].notna().any()]
    sc.pl.spatial(ad_sub, color=not_na_cols, na_color="lightgray", spot_size=10)
    break

## clean and split gloms

In [ ]:
from spatial_tcr.spatial import annotate_ccs, filter_ccs

ad_glom = adata[adata.obs["leiden"] == str(glom_clust)].copy()

# calc graph
npc.gc.construct_multi_sample_graph(ad_glom, radius=20, sample_key="sample")

annotate_ccs(ad_glom)

ad_glom = filter_ccs(ad_glom, min_cells=25)

In [ ]:
for s in ad_glom.obs["sample"].unique():
    print()
    ad_sub = ad_glom[ad_glom.obs["sample"] == s].copy()
    sc.pl.spatial(ad_sub, color="cc", spot_size=10)
    break
ad_sub.obs["cc"].value_counts()

In [ ]:
adata.obs["glom_annot"] = None
adata.obs.loc[ad_glom.obs_names, "glom_annot"] = "glom_" + ad_glom.obs["cc"].astype(str)
adata.obs["in_glom"] = adata.obs["glom_annot"].notna()

In [ ]:
for s in adata.obs["sample"].unique()[1::]:
    ad_sub = adata[adata.obs["sample"] == s].copy()
    sc.pl.spatial(
        ad_sub, color=["is_POD", "glom_annot"], na_color="lightgray", spot_size=10
    )
    break

In [ ]:
# # remove leiden cols
columns_to_drop = [c for c in adata.obs.columns if c.startswith("leiden_")]

print(columns_to_drop)

adata.obs.drop(columns=columns_to_drop, inplace=True)

In [ ]:
adata

In [ ]:
adata.write_h5ad("data/xenium/processed/05.1-kidney_tcr_nichepca.h5ad")